In [ ]:
import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
conn = sqlite3.connect("/Users/SOMNATH MAJHI/ML PROJECT/Data/inventory.db")

In [ ]:
tables = pd.read_sql_query("SELECT name from sqlite_master WHERE type='table'",conn)

In [ ]:
tables

In [ ]:
for table in tables['name']:
    print(f"Table Name: {table}")
    display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5",conn))

In [ ]:
pd.read_sql_query("select * from purchases LIMIT 5",conn)

In [ ]:
purchase_agg_df = pd.read_sql_query('''
select
p.PONumber,
count(DISTINCT p.Brand) as total_brands,
sum(p.Quantity) as total_item_quantity,
sum(p.Dollars) as total_item_dollars,
avg(julianday(p.ReceivingDate) - julianday(p.PODate)) as avg_receiving_delay
from purchases p
group by p.PONumber
''',conn)

In [ ]:
display(purchase_agg_df)

In [ ]:
#
pd.read_sql_query("""
select
vi.Quantity as invoice_quantity,
vi.Dollars as invoice_dollars,
vi.Freight,
(julianday(vi.InvoiceDate) - julianday(vi.PODate)) AS days_po_to_invoice,
(julianday(vi.PayDate) - julianday(vi.InvoiceDate)) AS days_to_pay
from vendor_invoice vi
""",conn)

In [ ]:
#JOINING THE TWO TABLES

df=pd.read_sql_query("""
WITH purchase_agg AS (
    select
        p.PONumber,
        count(DISTINCT p.Brand) as total_brands,
        sum(p.Quantity) as total_item_quantity,
        sum(p.Dollars) as total_item_dollars,
        avg(julianday(p.ReceivingDate) - julianday(p.PODate)) as avg_receiving_delay
    from purchases p
    group by p.PONumber
)

select
    vi.PONumber,
    vi.Quantity as invoice_quantity,
    vi.Dollars as invoice_dollars,
    vi.Freight,
    (julianday(vi.InvoiceDate) - julianday(vi.PODate)) AS days_po_to_invoice,
    (julianday(vi.PayDate) - julianday(vi.InvoiceDate)) AS days_to_pay,
    pa.total_brands,
    pa.total_item_quantity,
    pa.total_item_dollars,
    pa.avg_receiving_delay
    
FROM vendor_invoice vi
LEFT JOIN purchase_agg pa
ON vi.PONumber = pa.PONumber
""",conn)

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
def create_invoice_risk_label(row):

    #Invoice total mismatch with item-level total
    if(abs(row["invoice_dollars"] - row["total_item_dollars"]) > 5):
        return 1

    #Abnormally high receiving delay
    if row["avg_receiving_delay"] > 10:
        return 1

    return 0

df["flag_invoice"] = df.apply(create_invoice_risk_label, axis=1)
df["flag_invoice"].value_counts()

In [ ]:
df["flag_invoice"].value_counts().plot(kind = "bar")

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(df.corr(),annot=True)
plt.show()

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(df.iloc[:,1:-1].corr(),annot=True)
plt.show()

In [ ]:
flagged =  df[df["flag_invoice"] == 1]
normal = df[df["flag_invoice"] == 0]

In [ ]:
#
significant_features = []
non_significant_features = []
results = []

In [ ]:
metrics = [
    'invoice_quantity',
    'invoice_dollars',
    'Freight',
    'days_po_to_invoice',
    'days_to_pay',
    'total_brands',
    'total_item_quantity',
    'total_item_dollars',
    'avg_receiving_delay'
]

In [ ]:
from scipy.stats import ttest_ind
for metric in metrics:
    flagged_mean = flagged[metric].mean()
    normal_mean = normal[metric].mean()

    t_stat, p_value = ttest_ind(
        flagged[metric].dropna(),
        normal[metric].dropna(),
        equal_var = False
    )

    if p_value < 0.05 :
        significant_features.append(metric)
        results.append(
            {
                "metric":metric,
                "flagged_mean" :flagged_mean.round(2),
                "normal_mean": normal_mean.round(2),
                "p_value" : p_value.round(3)
            }
        ) 

    else:
        non_significant_features.append(metric)
        print(metric)
        print({
            "metric":metric,
                "flagged_mean" :float(flagged_mean.round(2)),
                "normal_mean": float(normal_mean.round(2)),
                "p_value" : float(p_value.round(3))
        })

In [ ]:
non_significant_features

In [ ]:
significant_features